[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rijifl/yolo-world/blob/main/notebooks/yolo_world_walkthrough.ipynb)

# YOLO-World: detect anything you can name, in real time

**Paper:** Tianheng Cheng, Lin Song, Yixiao Ge, Wenyu Liu, Xinggang Wang, Ying Shan. *YOLO-World: Real-Time Open-Vocabulary Object Detection.* CVPR 2024. [arXiv:2401.17270](https://arxiv.org/abs/2401.17270)
**Repo (demo, experiments, results):** https://github.com/Rijifl/yolo-world

A normal YOLO detector can only find the categories it was trained on (for COCO, 80 of them). YOLO-World keeps the fast YOLOv8 design but lets **you** choose the categories at run time, just by typing words. It has three parts: a YOLOv8 detector, a **frozen CLIP text encoder** that turns your words into vectors, and a new neck called **RepVL-PAN** that mixes the text and image features. Each predicted box gets an "object embedding", and its score for a word is a (scaled) similarity between that embedding and the word's text embedding, so classification becomes *matching boxes to words*. The paper calls the workflow **prompt-then-detect**: you encode your vocabulary once, offline, and then only the detector runs on each image. On the LVIS benchmark, the paper's YOLO-World-L reaches **35.4 AP (Fixed AP, LVIS minival) at 52.0 FPS on one V100 GPU without TensorRT**, after re-parameterization (the text encoder removed). With the text encoder still attached, the same model runs at 17.6 FPS.

In this notebook you will: (1) see a closed-set YOLOv8 fail on objects outside COCO, (2) fix it with YOLO-World by just typing words, (3) time "encode once" vs "encode every image", (4) see how prompt wording changes the scores, and (5) launch the live demo.

**Before you start:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Everything runs top to bottom without edits (about 3 to 5 minutes, mostly downloads). The last cell starts a live web demo and keeps running until you stop it.

## 1. Setup

We install the Ultralytics version this repo was tested with (8.4.166). YOLO-World's `set_classes()` needs OpenAI's CLIP text encoder; Ultralytics would install its CLIP fork automatically the first time, but we install it up front so "Run all" never stalls.

In [ ]:
%pip install -q ultralytics==8.4.166 ftfy regex
%pip install -q "git+https://github.com/ultralytics/CLIP.git@a13192f8cb767260d7dfd98c843b0716593169e7"

Next we get the sample images from the repo (8 CC-licensed COCO val2017 photos, see `demo/images/ATTRIBUTION.md`). If you run this from `notebooks/` inside a local copy of the repo, it uses that copy instead of cloning.

In [ ]:
import os, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/Rijifl/yolo-world.git"
REPO = Path("yolo-world")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if (Path("..") / "demo" / "images").exists():
    REPO = Path("..").resolve()  # running from notebooks/ inside a local copy of the repo
elif not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

IMAGES = REPO / "demo" / "images"
GUITAR = str(IMAGES / "guitar_stage.jpg")
BIKE = str(IMAGES / "bike_helmet.jpg")
print("Sample images:", sorted(p.name for p in IMAGES.glob("*.jpg")))

In [ ]:
import torch
import ultralytics

USE_GPU = torch.cuda.is_available()
DEVICE = 0 if USE_GPU else "cpu"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if USE_GPU else "CPU")
if not USE_GPU:
    print("No GPU found: everything still works, just slower. On Colab: Runtime -> Change runtime type -> T4 GPU.")


def weights(name):
    # Use the repo's cached weights if you run this locally after scripts/download_models.py;
    # otherwise Ultralytics downloads the file (this is what happens on Colab).
    local = REPO / "weights" / name
    return str(local) if local.exists() else name


if (REPO / "weights" / "clip" / "ViT-B-32.pt").exists():  # local runs only: reuse the cached CLIP file
    import ultralytics.nn.text_model as _tm
    _tm.WEIGHTS_DIR = REPO / "weights"

In [ ]:
from IPython.display import display
from PIL import Image


def show(result, width=640):
    # Draw an Ultralytics result (boxes + labels + scores) and show it inline.
    im = Image.fromarray(result.plot()[..., ::-1])  # plot() returns BGR
    im.thumbnail((width, width))
    display(im)


def summarize(result):
    names = result.names
    rows = sorted(((names[int(c)], float(p)) for c, p in zip(result.boxes.cls, result.boxes.conf)),
                  key=lambda t: -t[1])
    return ", ".join(f"{n} {p:.2f}" for n, p in rows) or "(nothing above the threshold)"

## 2. The problem: a closed-set detector

YOLOv8 trained on COCO has a fixed list of 80 classes. Whatever you show it, it can only answer with one of those 80 names. The paper puts it this way: "Once object categories are defined and labeled, trained detectors can only detect those specific categories."

Let's run YOLOv8-S on a concert photo. Is there a `guitar` or `microphone` class?

In [ ]:
from ultralytics import YOLO

yolov8 = YOLO(weights("yolov8s.pt"))
coco_names = list(yolov8.names.values())
print(f"YOLOv8 knows {len(coco_names)} classes:\n", ", ".join(coco_names), "\n")
for word in ["person", "guitar", "microphone", "helmet"]:
    print(f"  is '{word}' a COCO class? {word in coco_names}")

r = yolov8.predict(GUITAR, device=DEVICE, conf=0.25, verbose=False)[0]
print("\nYOLOv8-S found:", summarize(r))
show(r)

YOLOv8 can find the person, but it has no way to say "guitar" or "microphone": those words are simply not in its output layer. Adding them would mean collecting labeled boxes and retraining.

## 3. YOLO-World: just type the words

We load `yolov8s-worldv2.pt`, the small (S) YOLO-World checkpoint that Ultralytics ships. This is **YOLO-World v2**, a code release from the authors on 2024-02-29, a month after the arXiv preprint. v2 **removed the Image-Pooling Attention** module and **replaced the L2-Norm in the contrastive head with BatchNorm**, both to make deployment faster. So it is close to, but not exactly, the architecture described in the paper. Also, the paper's headline numbers are for the large (L) model; we use S because it is quick.

`set_classes([...])` runs the CLIP text encoder on your words once and stores the embeddings inside the model. After that, `predict()` only runs the detector.

In [ ]:
from ultralytics import YOLOWorld

world = YOLOWorld(weights("yolov8s-worldv2.pt"))
if USE_GPU:
    world.to("cuda")  # put the detector (and therefore the CLIP text encoder it loads) on the GPU

vocab = ["person", "guitar", "microphone"]
world.set_classes(vocab)  # first call also loads CLIP ViT-B/32 (a one-time download on Colab)
r = world.predict(GUITAR, device=DEVICE, conf=0.10, verbose=False)[0]
print("Vocabulary:", vocab)
print("YOLO-World-S found:", summarize(r))
show(r)

Same image, **new vocabulary, no retraining**: we just call `set_classes` again with different words. (We use a confidence threshold of 0.10 here, same as the live demo. If a word does not show up, its best box may just be below the threshold; try lowering `conf`.)

In [ ]:
vocab2 = ["shirt", "necktie", "hand", "hair"]
world.set_classes(vocab2)
r = world.predict(GUITAR, device=DEVICE, conf=0.10, verbose=False)[0]
print("Vocabulary:", vocab2)
print("YOLO-World-S found:", summarize(r))
show(r)

## 4. Mini-experiment: why "prompt-then-detect" matters

Older open-vocabulary detectors "simultaneously encode images and texts as input for prediction, which is time-consuming for practical applications" (paper, Fig. 2 caption). YOLO-World instead uses an **offline vocabulary**: "The offline vocabulary allows for avoiding computation for each input and provides the flexibility to adjust the vocabulary as needed" (Sec. 3.2).

What Ultralytics does (and what we measure here): **encode the vocabulary once, cache the embeddings, reuse them for every image.** The paper goes further and *re-parameterizes*: it folds the embeddings into the weights of RepVL-PAN (as 1x1 conv / linear layers), so the text encoder is not needed at all at deployment. This notebook does **not** demonstrate that folding step.

We time three things:
1. text encoding alone (`set_classes`),
2. detection per image with the cached embeddings,
3. detection per image when we (wastefully) re-encode the text for every image.

**These times depend on your GPU (or CPU) and are only illustrative. They are not the paper's numbers, which were measured on a V100 with the L model.**

In [ ]:
import statistics

N = 20 if USE_GPU else 5  # CPU is slow; fewer runs keep "Run all" short
words = ["person", "guitar", "microphone", "necktie", "helmet", "flag", "lamp", "traffic cone"]
img = GUITAR


def sync():
    if USE_GPU:
        torch.cuda.synchronize()


def ms(fn, n):
    out = []
    for _ in range(n):
        sync(); t0 = time.perf_counter()
        fn()
        sync(); out.append((time.perf_counter() - t0) * 1000)
    return statistics.median(out)


# (1) text encoding once. Warm up first so we do not time the one-off CLIP loading.
world.set_classes(words)
t_text = ms(lambda: world.model.set_classes(words), N)  # the step inside world.set_classes() that runs CLIP

# (2) detection with cached embeddings (after warm-up)
world.set_classes(words)
for _ in range(3):
    world.predict(img, device=DEVICE, verbose=False)
t_det = ms(lambda: world.predict(img, device=DEVICE, verbose=False), N)


# (3) re-encode the text before every image (what "encode images and texts together" costs)
def encode_then_detect():
    world.model.set_classes(words)  # same words -> same result, just recomputed
    world.predict(img, device=DEVICE, verbose=False)


t_both = ms(encode_then_detect, N)

print(f"Device: {torch.cuda.get_device_name(0) if USE_GPU else 'CPU'} | vocabulary of {len(words)} words | median of {N} runs")
print(f"  (1) CLIP text encoding, once per vocabulary : {t_text:7.1f} ms")
print(f"  (2) per image, cached text embeddings       : {t_det:7.1f} ms  (~{1000 / t_det:.0f} images/s)")
print(f"  (3) per image, re-encoding text every time  : {t_both:7.1f} ms  (~{1000 / t_both:.0f} images/s)")
print(f"  -> re-encoding makes each image {t_both / t_det:.2f}x slower here; with a cached vocabulary the text")
print(f"     cost is paid once, no matter how many images (or video frames) follow.")

Per-image times include reading the image file, Ultralytics' pre-processing and box post-processing (NMS), so they are wall-clock `predict()` times, not a pure forward pass. For a sense of scale on the paper's side: YOLO-World-L ran at 52.0 FPS re-parameterized vs. 17.6 FPS with the text encoder (V100, Table 2), for a very different model size, vocabulary (LVIS has 1203 classes) and hardware.

## 5. Prompt wording playground

YOLO-World does not understand words by itself: it compares each box's embedding with the **CLIP text embedding** of your prompt. Different wordings of the same object land at different points in CLIP's embedding space, so they can get different scores. The CLIP text encoder is kept **frozen** during YOLO-World's pre-training (the paper found frozen CLIP beat fine-tuning it), so the model inherits CLIP's sense of what words mean.

Below we ask for the same object, a bicycle helmet, with three wordings, one at a time. We don't tell you which one wins; look at the numbers.

In [ ]:
wordings = ["helmet", "bicycle helmet", "a protective hat"]
for w in wordings:
    world.set_classes([w])
    r = world.predict(BIKE, device=DEVICE, conf=0.01, verbose=False)[0]
    confs = sorted((float(c) for c in r.boxes.conf), reverse=True)
    best = f"{confs[0]:.3f}" if confs else "none"
    print(f"{w!r:22} best score {best:>6} | boxes above 0.10: {sum(c >= 0.10 for c in confs)}")

world.set_classes(["person", "bicycle", "helmet"])
show(world.predict(BIKE, device=DEVICE, conf=0.10, verbose=False)[0])

Things to try: plural vs. singular (`helmets`), adding a color (`red helmet`), or putting two competing words in one vocabulary (for example `["helmet", "hat"]`). Compare scores only *within* one vocabulary: the text embeddings also steer the network's text-guided layers, and each box keeps only its best-matching word, so changing the word list can change the scores.

## 6. Try it yourself

Edit `MY_WORDS` and `MY_IMAGE` (a file path or an `https://` image URL), then run the cell. Set `UPLOAD = True` to pick a photo from your computer (Colab only).

In [ ]:
MY_WORDS = ["person", "bicycle", "helmet", "sneakers", "backpack"]
MY_IMAGE = BIKE          # e.g. "https://ultralytics.com/images/bus.jpg" or a path
UPLOAD = False           # True -> Colab asks you to upload an image
MY_CONF = 0.10

if UPLOAD and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    MY_IMAGE = next(iter(uploaded))

world.set_classes(MY_WORDS)
r = world.predict(MY_IMAGE, device=DEVICE, conf=MY_CONF, verbose=False)[0]
print("Found:", summarize(r))
show(r)

## 7. Launch the class demo (runs until you stop it)

This starts the Gradio app from `demo/app.py`: YOLO-World and a closed-set YOLOv8 side by side on the same image, with your vocabulary. It first caches the S-size models and CLIP in the repo's `weights/` folder (`scripts/download_models.py --small`), then prints a public `https://....gradio.live` link you can open on your phone.

**This cell keeps running until you press the stop button** (that is normal; the web app lives inside it). It only runs on Colab; set `RUN_GRADIO = True` to run it elsewhere.

In [ ]:
RUN_GRADIO = IN_COLAB  # only launches on Colab by default

if not RUN_GRADIO:
    print("Skipping the live demo (not on Colab). Set RUN_GRADIO = True to launch it.")
else:
    %pip install -q gradio==6.29.0
    !cd "{REPO}" && python scripts/download_models.py --small && python demo/app.py --share --sizes S --no-browser

## Links

- Paper: [arXiv:2401.17270](https://arxiv.org/abs/2401.17270) (CVPR 2024)
- Official code: https://github.com/AILab-CVC/YOLO-World
- Ultralytics YOLO-World docs: https://docs.ultralytics.com/models/yolo-world/
- This repo: https://github.com/Rijifl/yolo-world and the experiment results: [results/README_results.md](https://github.com/Rijifl/yolo-world/blob/main/results/README_results.md)